# Validation Comparison for Two CFM Models

Evaluates trained models on the held-out test set and reports raw W1 plus normalized W1, where `normalized_w1 = W1 / std(real)` per morphology feature.

In [ ]:
from __future__ import annotations

import json
import math
import os
from dataclasses import dataclass
from pathlib import Path

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchdiffeq
from scipy.stats import wasserstein_distance

try:
    import wandb
except ImportError:
    wandb = None

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

## Config

Set the two checkpoint paths below. Use `architecture='pca'` for the PCA-conditioned model and `architecture='full'` for the full-expression model with an MLP gene encoder.

In [ ]:
@dataclass
class EvalConfig:
    data_path: str = "data/whole_dataset_test.h5ad"
    output_dir: str = "outputs/test_model_comparison"
    artifact_download_dir: str = "outputs/test_model_comparison/artifacts"
    leiden_key: str = "leiden"
    seed: int = 42
    y_dim: int = 56
    time_emb_dim: int = 128
    ode_rtol: float = 1e-5
    ode_atol: float = 1e-5
    std_eps: float = 1e-8


cfg = EvalConfig()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg.data_path = str(PROJECT_ROOT / cfg.data_path)
cfg.output_dir = str(PROJECT_ROOT / cfg.output_dir)
cfg.artifact_download_dir = str(PROJECT_ROOT / cfg.artifact_download_dir)
os.makedirs(cfg.output_dir, exist_ok=True)
os.makedirs(cfg.artifact_download_dir, exist_ok=True)

MODEL_SPECS = [
    {
        "name": "unconditioned",
        "architecture": "baseline",
        "checkpoint_path": "models/morphology/dark-feather-17.pt",
        "c_dim": 0,
        "hidden_dim": 512,
        "n_res_blocks": 6,
    },
    # {
    #     "name": "strat_pca_4_resblocks",
    #     "architecture": "pca",
    #     "checkpoint_path": "models/morphology/stratified_pca_baseline.pt",
    #     "c_dim": 50,
    #     "gene_encoder_hidden_dim": 512,
    #     "hidden_dim": 512,
    #     "n_res_blocks": 4,
    # },
    # W&B Artifact example. The notebook will read config.json and model_ema.pt from it.
    {
        "name": "50pc_film_model",
        "artifact": "hneumann-university-of-mannheim/cfm-morphology/morphology-pca-film-model:v0",
    },
    {
        "name": "50pc_concat_model",
        "artifact": "hneumann-university-of-mannheim/cfm-morphology/morphology-pca-concat-model:v0",
    },
    {
        "name": "full_film_model",
        "artifact": "hneumann-university-of-mannheim/cfm-morphology/morphology-full-film-model:v0",
    },
    {
        "name": "30pc_film",
        "artifact": "hneumann-university-of-mannheim/cfm-morphology/morphology-pca-film-model:v3",
    },
    # {
    #     "name": "30pc_film_50k",
    #     "artifact": "hneumann-university-of-mannheim/cfm-morphology/morphology-pca-film-model:v2",
    # },
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


def artifact_to_spec(spec: dict) -> dict:
    if wandb is None:
        raise ImportError("wandb is required to load model specs from W&B Artifacts")

    artifact_ref = spec["artifact"]
    print(f"Downloading W&B artifact {artifact_ref} ...")
    api = wandb.Api()
    artifact = api.artifact(artifact_ref, type="model")
    safe_artifact_dir = artifact_ref.replace("/", "__").replace(":", "__")
    artifact_dir = Path(
        artifact.download(root=str(Path(cfg.artifact_download_dir) / safe_artifact_dir))
    )

    config_path = artifact_dir / "config.json"
    checkpoint_path = artifact_dir / "model_ema.pt"
    y_stats_path = artifact_dir / "y_stats.pt"
    if not config_path.exists():
        raise FileNotFoundError(f"Artifact {artifact_ref} does not contain config.json")
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Artifact {artifact_ref} does not contain model_ema.pt"
        )
    if not y_stats_path.exists():
        raise FileNotFoundError(f"Artifact {artifact_ref} does not contain y_stats.pt")

    with open(config_path) as f:
        train_cfg = json.load(f)

    variant = train_cfg.get("model_variant")
    architecture_by_variant = {
        "pca_concat": "pca",
        "pca_film": "pca_film",
        "full_film": "film_full",
    }
    architecture = architecture_by_variant.get(variant, train_cfg.get("architecture"))
    if architecture is None:
        raise ValueError(f"Cannot infer architecture from artifact config: {train_cfg}")

    resolved = {
        "name": spec.get("name", artifact.name),
        "architecture": architecture,
        "checkpoint_path": str(checkpoint_path),
        "y_stats_path": str(y_stats_path),
        "artifact": artifact_ref,
        "model_variant": variant,
        "c_dim": int(train_cfg.get("c_dim", 50)),
        "gene_encoder_hidden_dim": int(train_cfg.get("gene_encoder_hidden_dim", 512)),
        "hidden_dim": int(train_cfg["hidden_dim"]),
        "n_res_blocks": int(train_cfg["n_res_blocks"]),
    }
    resolved.update({k: v for k, v in spec.items() if k not in resolved})
    return resolved


def resolve_model_specs(model_specs: list[dict]) -> list[dict]:
    resolved = []
    for spec in model_specs:
        resolved.append(artifact_to_spec(spec) if "artifact" in spec else spec)
    return resolved


def resolve_checkpoint_path(spec: dict) -> Path:
    path = Path(spec["checkpoint_path"])
    return path if path.is_absolute() else PROJECT_ROOT / path


MODEL_SPECS = resolve_model_specs(MODEL_SPECS)

## Model Definitions

In [ ]:
def sinusoidal_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    if t.dim() == 0:
        t = t.unsqueeze(0)
    half = dim // 2
    freqs = torch.exp(
        -math.log(10000)
        * torch.arange(half, dtype=torch.float32, device=t.device)
        / max(half - 1, 1)
    )
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResBlock(nn.Module):
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.SiLU(),
            nn.Linear(dim, dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.net(x)

class BaselineMorphologyVectorField(nn.Module):
    """v_θ(y_t, t, c) : R^56 × [0,1] × R^50 → R^56"""

    def __init__(
        self,
        y_dim: int,
        hidden_dim: int,
        n_res_blocks: int,
        time_emb_dim: int,
    ) -> None:
        super().__init__()
        self.time_emb_dim = time_emb_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim),
        )
        self.input_proj = nn.Linear(y_dim + time_emb_dim, hidden_dim)
        self.res_blocks = nn.ModuleList([ResBlock(hidden_dim) for _ in range(n_res_blocks)])
        self.output_head = nn.Linear(hidden_dim, y_dim)

    def forward(self, y_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_mlp(sinusoidal_embedding(t, self.time_emb_dim))
        x = self.input_proj(torch.cat([y_t, t_emb], dim=-1))
        for block in self.res_blocks:
            x = block(x)
        return self.output_head(x)

class PCAMorphologyVectorField(nn.Module):
    def __init__(self, y_dim: int, c_dim: int, hidden_dim: int, n_res_blocks: int, time_emb_dim: int) -> None:
        super().__init__()
        self.time_emb_dim = time_emb_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim),
        )
        self.input_proj = nn.Linear(y_dim + time_emb_dim + c_dim, hidden_dim)
        self.res_blocks = nn.ModuleList([ResBlock(hidden_dim) for _ in range(n_res_blocks)])
        self.output_head = nn.Linear(hidden_dim, y_dim)

    def forward(self, y_t: torch.Tensor, t: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_mlp(sinusoidal_embedding(t, self.time_emb_dim))
        x = self.input_proj(torch.cat([y_t, t_emb, c], dim=-1))
        for block in self.res_blocks:
            x = block(x)
        return self.output_head(x)


class FullInputMorphologyVectorField(nn.Module):
    def __init__(
        self,
        y_dim: int,
        x_dim: int,
        c_dim: int,
        gene_encoder_hidden_dim: int,
        hidden_dim: int,
        n_res_blocks: int,
        time_emb_dim: int,
    ) -> None:
        super().__init__()
        self.time_emb_dim = time_emb_dim
        self.gene_encoder = nn.Sequential(
            nn.Linear(x_dim, gene_encoder_hidden_dim),
            nn.LayerNorm(gene_encoder_hidden_dim),
            nn.SiLU(),
            nn.Linear(gene_encoder_hidden_dim, c_dim),
        )
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim),
        )
        self.input_proj = nn.Linear(y_dim + time_emb_dim + c_dim, hidden_dim)
        self.res_blocks = nn.ModuleList([ResBlock(hidden_dim) for _ in range(n_res_blocks)])
        self.output_head = nn.Linear(hidden_dim, y_dim)

    def forward(self, y_t: torch.Tensor, t: torch.Tensor, x_gene: torch.Tensor) -> torch.Tensor:
        c = self.gene_encoder(x_gene)
        t_emb = self.time_mlp(sinusoidal_embedding(t, self.time_emb_dim))
        x = self.input_proj(torch.cat([y_t, t_emb, c], dim=-1))
        for block in self.res_blocks:
            x = block(x)
        return self.output_head(x)

class FiLMResBlock(nn.Module):
    def __init__(self, hidden_dim: int, cond_dim: int) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.linear1 = nn.Linear(hidden_dim, hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, hidden_dim)
        self.film = nn.Linear(cond_dim, hidden_dim * 2)

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        gamma, beta = self.film(cond).chunk(2, dim=-1)
        h = self.linear1(torch.nn.functional.silu(self.norm1(x)))
        h = (1 + gamma) * h + beta
        h = self.linear2(torch.nn.functional.silu(self.norm2(h)))
        return x + h


class PCAFiLMMorphologyVectorField(nn.Module):
    def __init__(self, y_dim: int, c_dim: int, hidden_dim: int, n_res_blocks: int, time_emb_dim: int) -> None:
        super().__init__()
        self.time_emb_dim = time_emb_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim),
        )
        film_cond_dim = c_dim + time_emb_dim
        self.input_proj = nn.Linear(y_dim, hidden_dim)
        self.res_blocks = nn.ModuleList([
            FiLMResBlock(hidden_dim, film_cond_dim) for _ in range(n_res_blocks)
        ])
        self.output_norm = nn.LayerNorm(hidden_dim)
        self.output_head = nn.Linear(hidden_dim, y_dim)

    def forward(self, y_t: torch.Tensor, t: torch.Tensor, x_pca: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_mlp(sinusoidal_embedding(t, self.time_emb_dim))
        cond = torch.cat([x_pca, t_emb], dim=-1)
        x = self.input_proj(y_t)
        for block in self.res_blocks:
            x = block(x, cond)
        return self.output_head(self.output_norm(x))


class FullInputFiLMMorphologyVectorField(nn.Module):
    def __init__(
        self,
        y_dim: int,
        x_dim: int,
        c_dim: int,
        gene_encoder_hidden_dim: int,
        hidden_dim: int,
        n_res_blocks: int,
        time_emb_dim: int,
    ) -> None:
        super().__init__()
        self.time_emb_dim = time_emb_dim
        self.gene_encoder = nn.Sequential(
            nn.Linear(x_dim, gene_encoder_hidden_dim),
            nn.LayerNorm(gene_encoder_hidden_dim),
            nn.SiLU(),
            nn.Linear(gene_encoder_hidden_dim, c_dim),
        )
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim),
        )
        film_cond_dim = c_dim + time_emb_dim
        self.input_proj = nn.Linear(y_dim, hidden_dim)
        self.res_blocks = nn.ModuleList([
            FiLMResBlock(hidden_dim, film_cond_dim) for _ in range(n_res_blocks)
        ])
        self.output_norm = nn.LayerNorm(hidden_dim)
        self.output_head = nn.Linear(hidden_dim, y_dim)

    def forward(self, y_t: torch.Tensor, t: torch.Tensor, x_gene: torch.Tensor) -> torch.Tensor:
        c = self.gene_encoder(x_gene)
        t_emb = self.time_mlp(sinusoidal_embedding(t, self.time_emb_dim))
        cond = torch.cat([c, t_emb], dim=-1)
        x = self.input_proj(y_t)
        for block in self.res_blocks:
            x = block(x, cond)
        return self.output_head(self.output_norm(x))


## Data Loading and Validation Split

In [ ]:
_DROP_COLUMNS = {
    "Center_X", "Center_Y",
    "BoundingBoxMinimum_X", "BoundingBoxMaximum_X",
    "BoundingBoxMinimum_Y", "BoundingBoxMaximum_Y",
    "Orientation",
    "NormalizedMoment_1_0", "NormalizedMoment_0_0", "NormalizedMoment_0_1",
}


def get_morphology_columns(obs_columns: list[str]) -> list[str]:
    cols = [c for c in obs_columns if c[0].isupper()]
    cols = [c for c in cols if c not in _DROP_COLUMNS]
    cols = [c for c in cols if not c.startswith("SpatialMoment_")]
    assert len(cols) == cfg.y_dim, f"Expected {cfg.y_dim} morphological features, got {len(cols)}"
    return cols


def to_dense_float32(x) -> np.ndarray:
    if hasattr(x, "toarray"):
        x = x.toarray()
    return np.asarray(x, dtype=np.float32)


print(f"Loading {cfg.data_path} ...")
adata = anndata.read_h5ad(cfg.data_path)
morph_cols = get_morphology_columns(list(adata.obs.columns))
leiden_test = adata.obs[cfg.leiden_key].astype(str).to_numpy()

y_raw = torch.tensor(adata.obs[morph_cols].values.astype(np.float32), dtype=torch.float32)
nan_mask = torch.isnan(y_raw)
if nan_mask.any():
    for j in range(y_raw.shape[1]):
        col = y_raw[:, j]
        median = col[~torch.isnan(col)].median()
        y_raw[:, j] = torch.where(torch.isnan(col), median, col)
inf_mask = torch.isinf(y_raw)
if inf_mask.any():
    for j in range(y_raw.shape[1]):
        col = y_raw[:, j]
        finite_vals = col[torch.isfinite(col)]
        median = finite_vals.median()
        y_raw[:, j] = torch.where(torch.isinf(col), median, col)

y_log_test = torch.sign(y_raw) * torch.log1p(torch.abs(y_raw))

# Keep these aliases so the architecture-specific helper code can stay unchanged.
leiden_val = leiden_test
pca_val = torch.tensor(np.asarray(adata.obsm["X_pca"], dtype=np.float32), dtype=torch.float32)
full_val = torch.tensor(to_dense_float32(adata.X), dtype=torch.float32)
x_dim = full_val.shape[1]

print(f"Test size: {len(y_log_test):,}")
print(f"PCA input: {tuple(pca_val.shape)} | full input: {tuple(full_val.shape)} | y: {tuple(y_log_test.shape)}")

## Evaluation Helpers

In [ ]:
def build_model(spec: dict) -> nn.Module:
    if spec["architecture"] == "pca":
        return PCAMorphologyVectorField(
            y_dim=cfg.y_dim,
            c_dim=spec["c_dim"],
            hidden_dim=spec["hidden_dim"],
            n_res_blocks=spec["n_res_blocks"],
            time_emb_dim=cfg.time_emb_dim,
        )
    if spec["architecture"] == "full":
        return FullInputMorphologyVectorField(
            y_dim=cfg.y_dim,
            x_dim=x_dim,
            c_dim=spec["c_dim"],
            gene_encoder_hidden_dim=spec["gene_encoder_hidden_dim"],
            hidden_dim=spec["hidden_dim"],
            n_res_blocks=spec["n_res_blocks"],
            time_emb_dim=cfg.time_emb_dim,
        )
    if spec["architecture"] == "pca_film":
        return PCAFiLMMorphologyVectorField(
            y_dim=cfg.y_dim,
            c_dim=spec["c_dim"],
            hidden_dim=spec["hidden_dim"],
            n_res_blocks=spec["n_res_blocks"],
            time_emb_dim=cfg.time_emb_dim,
        )
    if spec["architecture"] == "film_full":
        return FullInputFiLMMorphologyVectorField(
            y_dim=cfg.y_dim,
            x_dim=x_dim,
            c_dim=spec["c_dim"],
            gene_encoder_hidden_dim=spec["gene_encoder_hidden_dim"],
            hidden_dim=spec["hidden_dim"],
            n_res_blocks=spec["n_res_blocks"],
            time_emb_dim=cfg.time_emb_dim,
        )
    if spec["architecture"] == "baseline":
        return BaselineMorphologyVectorField(
            y_dim=cfg.y_dim,
            hidden_dim=spec["hidden_dim"],
            n_res_blocks=spec["n_res_blocks"],
            time_emb_dim=cfg.time_emb_dim,
        )
    raise ValueError(f"Unknown architecture: {spec['architecture']}")


def validation_inputs_for(spec: dict) -> torch.Tensor | None:
    if spec["architecture"] in {"pca", "pca_film"}:
        c_dim = int(spec.get("c_dim", pca_val.shape[1]))
        if c_dim > pca_val.shape[1]:
            raise ValueError(f"{spec['name']} requests {c_dim} PCs, but pca_val has only {pca_val.shape[1]}")
        return pca_val[:, :c_dim]
    if spec["architecture"] in {"full", "film_full"}:
        return full_val
    if spec["architecture"] == "baseline":
        return None
    raise ValueError(f"Unknown architecture: {spec['architecture']}")


def infer_architecture_from_state_dict(state: dict[str, torch.Tensor]) -> str:
    has_gene_encoder = any(k.startswith("gene_encoder.") for k in state)
    has_film = any(".film." in k for k in state)
    if has_gene_encoder and has_film:
        return "film_full"
    if has_gene_encoder:
        return "full"
    if has_film:
        return "pca_film"
    input_width = int(state["input_proj.weight"].shape[1])
    if input_width == cfg.y_dim + cfg.time_emb_dim:
        return "baseline"
    return "pca"


def resolve_y_stats_path(spec: dict, checkpoint_path: Path) -> Path:
    if spec.get("y_stats_path") is not None:
        path = Path(spec["y_stats_path"])
        path = path if path.is_absolute() else PROJECT_ROOT / path
        if path.exists():
            return path
        raise FileNotFoundError(f"Configured y_stats_path does not exist: {path}")

    candidates = [
        checkpoint_path.parent / "y_stats.pt",
        PROJECT_ROOT / "outputs" / "y_stats.pt",
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"No y_stats.pt found for {spec['name']}. Add 'y_stats_path' to MODEL_SPECS "
        f"or place y_stats.pt next to {checkpoint_path.name}."
    )


def load_y_stats(spec: dict, checkpoint_path: Path) -> tuple[torch.Tensor, torch.Tensor]:
    y_stats_path = resolve_y_stats_path(spec, checkpoint_path)
    stats = torch.load(y_stats_path, map_location="cpu")
    stats_columns = list(stats.get("columns", []))
    if stats_columns and stats_columns != morph_cols:
        raise ValueError(
            f"Morphology columns in {y_stats_path} do not match the test data columns."
        )
    y_mean_model = stats["mean"].float()
    y_std_model = stats["std"].float().clamp(min=cfg.std_eps)
    return y_mean_model, y_std_model


def normalized_test_targets(y_mean_model: torch.Tensor, y_std_model: torch.Tensor) -> torch.Tensor:
    return (y_log_test - y_mean_model) / y_std_model


@torch.no_grad()
def sample_model(
    model: nn.Module,
    y_real_norm: torch.Tensor,
    y_mean_model: torch.Tensor,
    y_std_model: torch.Tensor,
    cond: torch.Tensor | None = None,
):
    model.eval()
    n = len(y_real_norm)
    cond = None if cond is None else cond.to(device)
    y0 = torch.randn(n, cfg.y_dim, device=device)

    def ode_fn(t: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        t_batch = t.expand(y.shape[0])
        if cond is None:
            return model(y, t_batch)
        return model(y, t_batch, cond)

    traj = torchdiffeq.odeint(
        ode_fn,
        y0,
        torch.tensor([0.0, 1.0], device=device),
        method="dopri5",
        rtol=cfg.ode_rtol,
        atol=cfg.ode_atol,
        options={"dtype": torch.float32},
    )
    y_gen_norm = traj[-1].cpu()
    y_mean_model = y_mean_model.cpu()
    y_std_model = y_std_model.cpu()
    y_real_log = y_real_norm.cpu() * y_std_model + y_mean_model
    y_gen_log = y_gen_norm * y_std_model + y_mean_model
    y_real = (torch.sign(y_real_log) * torch.expm1(torch.abs(y_real_log))).numpy()
    y_gen = (torch.sign(y_gen_log) * torch.expm1(torch.abs(y_gen_log))).numpy()
    return y_real, y_gen


def compute_w1_metrics(y_real: np.ndarray, y_gen: np.ndarray) -> pd.DataFrame:
    rows = []
    for i, col in enumerate(morph_cols):
        real_std = float(np.std(y_real[:, i]))
        w1 = float(wasserstein_distance(y_real[:, i], y_gen[:, i]))
        rows.append(
            {
                "feature": col,
                "real_mean": float(np.mean(y_real[:, i])),
                "gen_mean": float(np.mean(y_gen[:, i])),
                "real_std": real_std,
                "gen_std": float(np.std(y_gen[:, i])),
                "w1": w1,
                "normalized_w1": w1 / max(real_std, cfg.std_eps),
            }
        )
    return pd.DataFrame(rows)


eval_idx = np.arange(len(y_log_test))
print(f"Evaluating {len(eval_idx):,} test cells across Leiden clusters")

## Run Comparison

In [ ]:
summary_rows = []
feature_tables = []

for spec in MODEL_SPECS:
    checkpoint_path = resolve_checkpoint_path(spec)
    if not checkpoint_path.exists():
        print(f"Skipping {spec['name']}: checkpoint not found at {checkpoint_path}")
        continue

    state = torch.load(checkpoint_path, map_location=device)
    inferred_architecture = infer_architecture_from_state_dict(state)
    eval_spec = dict(spec)
    if inferred_architecture != spec["architecture"]:
        print(
            f"Warning: {spec['name']} spec says architecture={spec['architecture']!r}, "
            f"but checkpoint keys look like {inferred_architecture!r}. Using checkpoint architecture."
        )
        eval_spec["architecture"] = inferred_architecture

    print(f"\nEvaluating {eval_spec['name']} ({eval_spec['architecture']}) from {checkpoint_path}")
    model = build_model(eval_spec).to(device)
    model.load_state_dict(state)

    cond_all = validation_inputs_for(eval_spec)
    cond_val = None if cond_all is None else cond_all[eval_idx]
    y_real, y_gen = sample_model(
        model,
        y_test_norm[eval_idx],
        y_mean_model,
        y_std_model,
        cond=cond_val,
    )
    metrics = compute_w1_metrics(y_real, y_gen)
    metrics.insert(0, "model", eval_spec["name"])
    feature_tables.append(metrics)
    summary_rows.append({
        "model": eval_spec["name"],
        "architecture": eval_spec["architecture"],
        "source": spec.get("artifact", str(checkpoint_path)),
        "n_eval": int(len(eval_idx)),
        "mean_w1": float(metrics["w1"].mean()),
        "median_w1": float(metrics["w1"].median()),
        "mean_normalized_w1": float(metrics["normalized_w1"].mean()),
        "median_normalized_w1": float(metrics["normalized_w1"].median()),
    })

summary_df = (
    pd.DataFrame(summary_rows).sort_values("mean_normalized_w1")
    if summary_rows else
    pd.DataFrame(columns=["model", "architecture", "source", "n_eval", "mean_w1", "median_w1", "mean_normalized_w1", "median_normalized_w1"])
)
feature_df = pd.concat(feature_tables, ignore_index=True) if feature_tables else pd.DataFrame()

summary_path = Path(cfg.output_dir) / "test_model_summary.csv"
feature_path = Path(cfg.output_dir) / "test_feature_w1.csv"
summary_df.to_csv(summary_path, index=False)
feature_df.to_csv(feature_path, index=False)

display(summary_df)
print(f"Saved {summary_path}")
print(f"Saved {feature_path}")

## Plots

In [ ]:
if not summary_df.empty:
    summary_df_sorted = summary_df.sort_values("mean_normalized_w1")
    colors = plt.cm.tab20(np.random.permutation(len(summary_df)))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(summary_df_sorted["model"], summary_df_sorted["mean_normalized_w1"].sort_values(), color=colors)
    ax.set_ylabel("Mean normalized W1")
    ax.set_title("Test Comparison")
    ax.tick_params(axis="x", rotation=20)
    plt.tight_layout()
    path = Path(cfg.output_dir) / "test_model_summary.png"
    plt.savefig(path, dpi=140)
    plt.show()
    print(f"Saved {path}")

if not feature_df.empty:
    pivot = feature_df.pivot(index="feature", columns="model", values="normalized_w1")
    display(pivot.sort_values(pivot.columns[0], ascending=False).head(15))

In [ ]:
FEATURE_GROUPS = {
    "Size": [
        "Area", "BoundingBoxArea", "ConvexArea", "EquivalentDiameter",
        "Perimeter", "PerimeterCrofton", "MajorAxisLength", "MinorAxisLength",
        "MaximumRadius", "MeanRadius", "MedianRadius", "FilledArea",
    ],
    "Shape": [
        "Eccentricity", "FormFactor", "Extent", "Solidity", "Compactness",
        "EulerNumber",
    ],
    "Central Moments": [
        "CentralMoment_0_0", "CentralMoment_0_1", "CentralMoment_0_2",
        "CentralMoment_0_3", "CentralMoment_1_0", "CentralMoment_1_1",
        "CentralMoment_1_2", "CentralMoment_1_3", "CentralMoment_2_0",
        "CentralMoment_2_1", "CentralMoment_2_2", "CentralMoment_2_3",
    ],
    "Normalized Moments": [
        "NormalizedMoment_0_2", "NormalizedMoment_0_3",
        "NormalizedMoment_1_1", "NormalizedMoment_1_2",
        "NormalizedMoment_1_3", "NormalizedMoment_2_0",
        "NormalizedMoment_2_1", "NormalizedMoment_2_2",
        "NormalizedMoment_2_3", "NormalizedMoment_3_0",
        "NormalizedMoment_3_1", "NormalizedMoment_3_2",
        "NormalizedMoment_3_3",
    ],
    "Hu Moments": [
        "HuMoment_0", "HuMoment_1", "HuMoment_2", "HuMoment_3",
        "HuMoment_4", "HuMoment_5", "HuMoment_6",
    ],
    "Inertia": [
        "InertiaTensor_0_0", "InertiaTensor_0_1",
        "InertiaTensor_1_0", "InertiaTensor_1_1",
        "InertiaTensorEigenvalues_0", "InertiaTensorEigenvalues_1",
    ],
}


In [ ]:
# Compare models by morphological feature groups

feature_to_group = {
    feature: group
    for group, features in FEATURE_GROUPS.items()
    for feature in features
}

grouped_feature_df = feature_df.copy()
grouped_feature_df["feature_group"] = grouped_feature_df["feature"].map(feature_to_group)
grouped_feature_df["feature_group"] = grouped_feature_df["feature_group"].fillna("Other")

group_summary = (
    grouped_feature_df
    .groupby(["model", "feature_group"], as_index=False)
    .agg(
        n_features=("feature", "nunique"),
        mean_w1=("w1", "mean"),
        median_w1=("w1", "median"),
        mean_normalized_w1=("normalized_w1", "mean"),
        median_normalized_w1=("normalized_w1", "median"),
    )
    .sort_values(["feature_group", "mean_normalized_w1"])
)

display(group_summary)

In [ ]:
group_pivot = group_summary.pivot(
    index="feature_group",
    columns="model",
    values="mean_normalized_w1",
)

display(group_pivot)

In [ ]:
ax = group_pivot.plot(
    kind="bar",
    figsize=(12, 5),
    width=0.8,
)

ax.set_ylabel("Mean normalized W1")
ax.set_xlabel("Morphology feature group")
ax.set_title("Validation performance by morphology feature group")
ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
FEATURES_TO_COMPARE = ['Area', 'Eccentricity', 'HuMoment_0', 'InertiaTensorEigenvalues_1']


In [ ]:
# 2x2 histogram comparison:
# real vs unconditioned vs best model (50pc_film)

FEATURES_TO_PLOT = [
    "Area",
    "Eccentricity",
    "HuMoment_0",
    "InertiaTensorEigenvalues_1",
]

UNCONDITIONED_MODEL_NAME = "unconditioned"
BEST_MODEL_NAME = "50pc_film_model"   # <- uses your MODEL_SPECS name


def get_spec_by_name(name: str, model_specs: list[dict]) -> dict:
    matches = [spec for spec in model_specs if spec["name"] == name]
    if not matches:
        raise ValueError(f"No model spec found with name={name!r}")
    return matches[0]


def load_and_sample_spec(spec: dict):
    checkpoint_path = resolve_checkpoint_path(spec)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    state = torch.load(checkpoint_path, map_location=device)
    inferred_architecture = infer_architecture_from_state_dict(state)

    eval_spec = dict(spec)
    if inferred_architecture != eval_spec["architecture"]:
        print(
            f"Warning: {eval_spec['name']} spec says architecture={eval_spec['architecture']!r}, "
            f"but checkpoint keys look like {inferred_architecture!r}. Using checkpoint architecture."
        )
        eval_spec["architecture"] = inferred_architecture

    model = build_model(eval_spec).to(device)
    model.load_state_dict(state)

    y_mean_model, y_std_model = load_y_stats(eval_spec, checkpoint_path)
    y_test_norm = normalized_test_targets(y_mean_model, y_std_model)

    y_mean_model, y_std_model = load_y_stats(eval_spec, checkpoint_path)
    y_test_norm = normalized_test_targets(y_mean_model, y_std_model)

    cond_all = validation_inputs_for(eval_spec)
    cond_val = None if cond_all is None else cond_all[eval_idx]

    y_real_arr, y_gen_arr = sample_model(
        model,
        y_test_norm[eval_idx],
        y_mean_model,
        y_std_model,
        cond=cond_val,
    )

    return y_real_arr, y_gen_arr, eval_spec


uncond_spec = get_spec_by_name(UNCONDITIONED_MODEL_NAME, MODEL_SPECS)
best_spec = get_spec_by_name(BEST_MODEL_NAME, MODEL_SPECS)

y_real_uncond, y_gen_uncond, uncond_eval_spec = load_and_sample_spec(uncond_spec)
y_real_best, y_gen_best, best_eval_spec = load_and_sample_spec(best_spec)


In [ ]:
y_real_plot = y_real_best

real_color = "#2563EB"# blue
uncond_color = "#9CA3AF"    # gray
best_color = "#DC2626"      # vermillion/orange-red
colors = {"Real": "#2563EB", "Unconditioned": "#9CA3AF", "PCA+FiLM (30)": "#DC2626"}

feature_to_idx = {feature: i for i, feature in enumerate(morph_cols)}

missing = [f for f in FEATURES_TO_PLOT if f not in feature_to_idx]
if missing:
    raise ValueError(f"Missing features in morph_cols: {missing}")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

for ax, feature in zip(axes, FEATURES_TO_PLOT):
    idx = feature_to_idx[feature]

    real_vals = y_real_plot[:, idx]
    uncond_vals = y_gen_uncond[:, idx]
    best_vals = y_gen_best[:, idx]

    finite_mask = (
        np.isfinite(real_vals)
        & np.isfinite(uncond_vals)
        & np.isfinite(best_vals)
    )

    real_vals = real_vals[finite_mask]
    uncond_vals = uncond_vals[finite_mask]
    best_vals = best_vals[finite_mask]

    all_vals = np.concatenate([real_vals, uncond_vals, best_vals])
    lo, hi = np.percentile(all_vals, [1, 99])
    bins = np.linspace(lo, hi, 50)

    ax.hist(
        real_vals,
        bins=bins,
        density=True,
        alpha=0.45,
        color=real_color,
        label="real",
    )
    ax.hist(
        uncond_vals,
        bins=bins,
        density=True,
        alpha=0.45,
        color=uncond_color,
        label="unconditioned",
    )
    ax.hist(
        best_vals,
        bins=bins,
        density=True,
        alpha=0.45,
        color=best_color,
        label="30pc FiLM",
    )

    ax.set_title(feature)
    ax.set_ylabel("Density")
    ax.tick_params(axis="x", rotation=30)

axes[0].legend()
fig.suptitle("Morphology distributions: real vs unconditioned vs best model", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import gaussian_kde

features = [
    "Area",
    "Eccentricity",
    "HuMoment_0",
    "InertiaTensorEigenvalues_1",
]

real_color = "#2563EB"
uncond_color = "#6B7280"
best_color = "#DC2626"

y_real_df = pd.DataFrame(y_real_plot, columns=morph_cols)
y_uncond_df = pd.DataFrame(y_gen_uncond, columns=morph_cols)
y_best_df = pd.DataFrame(y_gen_best, columns=morph_cols)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

for ax, feat in zip(axes, features):
    real_vals = y_real_df[feat].replace([np.inf, -np.inf], np.nan).dropna()
    uncond_vals = y_uncond_df[feat].replace([np.inf, -np.inf], np.nan).dropna()
    best_vals = y_best_df[feat].replace([np.inf, -np.inf], np.nan).dropna()

    lo, hi = real_vals.quantile([0.005, 0.995])
    x_range = np.linspace(lo, hi, 300)

    kde_real = gaussian_kde(real_vals)
    kde_uncond = gaussian_kde(uncond_vals)
    kde_best = gaussian_kde(best_vals)

    ax.plot(
        x_range,
        kde_real(x_range),
        label="Real",
        color=real_color,
        linewidth=2.5,
    )
    # ax.plot(
    #     x_range,
    #     kde_uncond(x_range),
    #     label="Unconditioned",
    #     color=uncond_color,
    #     linewidth=2.5,
    #     linestyle=":",
    # )
    ax.plot(
        x_range,
        kde_best(x_range),
        label="Best model",
        color=best_color,
        linewidth=2.5,
        linestyle="--",
    )

    ax.set_title(feat, fontsize=14)
    ax.legend(fontsize=11)
    ax.set_ylabel("")

plt.tight_layout()
plt.show()